# 06a. Sequence model

A small transformer that reads each transaction's recent history: the last transactions of the sender and of the receiver, alongside the current row's 22 features. Its purpose is to test whether that history carries signal the 22 features miss, in particular for laundering from hub senders, which every model so far fails to detect. The design was fixed before any results (see the PRD). This notebook first measures how much history accounts actually have, to set the sequence length.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from bankguard.data import load_transactions, split_by_date
from bankguard.features import BIG, account_codes, count_prior, minutes_from_start

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
val = split_by_date(tx)["validation"]

sender, receiver = account_codes(tx)
minutes = minutes_from_start(tx)
self_tx = tx["self_transfer"].to_numpy()

# Every transaction is one event for the sender and one for the receiver.
# A self transfer is the same account on both sides, so it counts once.
event_codes = np.concatenate([sender, receiver[~self_tx]])
event_minutes = np.concatenate([minutes, minutes[~self_tx]])

pos = val.index.to_numpy()
history = pd.DataFrame({
    "sender": count_prior(event_codes, event_minutes, sender[pos], minutes[pos], BIG),
    "receiver": count_prior(event_codes, event_minutes, receiver[pos], minutes[pos], BIG),
    "laundering": val["is_laundering"].to_numpy(),
})

quantiles = [0.5, 0.75, 0.9, 0.99]
summary = history.groupby("laundering")[["sender", "receiver"]].quantile(quantiles).unstack(0)
print("share of validation rows with at least 32 earlier transactions:")
print((history[["sender", "receiver"]] >= 32).mean().round(3).to_dict())
summary

share of validation rows with at least 32 earlier transactions:
{'sender': 0.819, 'receiver': 0.541}


sender            receiver       
laundering         0          1        0      1
0.50            59.0      29.00     33.0   24.0
0.75            94.0      67.00     50.0   44.0
0.90          6169.3   10211.00     72.0   80.0
0.99        129610.0  129535.15    127.0  142.3

Most accounts have short histories: the median validation row has 59 earlier transactions on the sender side and 33 on the receiver side, and laundering rows have fewer (29 and 24). About 10% of senders are hub accounts with thousands to 130,000 earlier transactions in at most eight days, so no fixed sequence length can hold their full history. A length of 64 is used for both sides: it covers the full history for about three quarters of laundering rows on each side, at a modest cost for a small model, while longer windows would add little before reaching the hub accounts. For hub senders the model therefore sees only their most recent 64 transactions, a span of minutes; if hub sent laundering stays undetected, this limit is the likely reason.